# Background Event Generation

Generates background events (`qq -> qq aa`) separately from signal.
This process is much slower due to many QCD diagrams.

Run this notebook independently (or on a different machine) and then
uncomment the background `add_sample` in `semi_parametric_setup.ipynb`.

In [ ]:
import subprocess
subprocess.check_call(["pip", "install", "-e", "/home/shared/madminer"])

In [ ]:
import os

os.environ["LD_LIBRARY_PATH"] = (
    "/madminer/software/MG5_aMC_v2_9_16/HEPTools/lhapdf6_py3/lib:"
    + os.environ.get("LD_LIBRARY_PATH", "")
)

In [ ]:
# ============================================================
# BATCH CONFIGURATION - must match the signal notebook
# ============================================================
BATCH_ID = 1

import logging
from madminer.core import MadMiner

mg_dir = os.getenv("MG_FOLDER_PATH")

logging.basicConfig(
    format="%(asctime)-5.5s %(name)-20.20s %(levelname)-7.7s %(message)s",
    datefmt="%H:%M",
    level=logging.INFO,
)
for key in logging.Logger.manager.loggerDict:
    if "madminer" not in key:
        logging.getLogger(key).setLevel(logging.WARNING)

print(f"Batch ID: {BATCH_ID}")

In [ ]:
# Load the same setup as the signal notebook
miner = MadMiner()
miner.load("data/setup_semi_parametric.h5")

## Generate Background Events

The background process (`qq -> qq aa`) has QCD vertices, so mu_R scale
variations will have a real effect here (unlike the EW-only signal).

Change the run card for more events. This cell will take a long time.

In [ ]:
mg_systematics = ["scale_corr"]

miner.run(
    sample_benchmark="sm",
    mg_directory=mg_dir,
    mg_process_directory=f"./mg_processes/bkg_semi_b{BATCH_ID}",
    proc_card_file="cards/proc_card_background.dat",
    param_card_template_file="cards/param_card_template.dat",
    run_card_file="cards/run_card_background_small.dat",
    log_directory=f"logs/bkg_semi_b{BATCH_ID}",
    systematics=mg_systematics,
)

## Verify

Quick check that the LHE file was produced and has scale variation weights.

In [ ]:
import gzip
from xml.etree import ElementTree as ET

lhe_path = f"mg_processes/bkg_semi_b{BATCH_ID}/Events/run_01/unweighted_events.lhe.gz"

with gzip.open(lhe_path, 'rt') as f:
    content = f.read()

start = content.index('<initrwgt>')
end = content.index('</initrwgt>') + len('</initrwgt>')
initrwgt = ET.fromstring(content[start:end])

for wg in initrwgt.findall('weightgroup'):
    name = wg.attrib.get('name', 'UNNAMED')
    if 'scale' in name.lower():
        print(f'Weight group: "{name}"')
        for w in wg.findall('weight'):
            a = dict(w.attrib)
            print(f'  id={a.get("id","?"):5s}  MUR={a.get("MUR","-")}  MUF={a.get("MUF","-")}')

print(f"\nBackground LHE file ready at: {lhe_path}")
print(f"Now uncomment the background add_sample in semi_parametric_setup.ipynb and re-run from cell 14.")